In [9]:
import pandas as pd
import sqlite3

In [3]:
connection = sqlite3.connect('../../data/checking-logs.sqlite')

In [4]:
results = pd.read_sql('SELECT * FROM control', connection)
results

,uid,labname,first_commit_ts,first_view_ts
0,user_11,laba05,2020-05-03 21:06:55.970293,2020-04-27 00:40:05.761783
1,user_11,project1,2020-05-03 23:45:33.673409,2020-04-27 00:40:05.761783
2,user_12,laba04,2020-04-18 17:07:51.767358,2020-04-27 00:40:05.761783
3,user_12,laba04s,2020-04-26 15:42:38.070593,2020-04-27 00:40:05.761783
4,user_12,laba05,2020-05-03 08:39:25.174316,2020-04-27 00:40:05.761783
...,...,...,...,...
76,user_8,laba04s,2020-04-19 10:22:35.761944,2020-04-27 00:40:05.761783
77,user_8,laba05,2020-05-02 13:28:07.705193,2020-04-27 00:40:05.761783
78,user_8,laba06,2020-05-16 17:56:15.755553,2020-04-27 00:40:05.761783
79,user_8,laba06s,2020-05-16 20:01:07.900727,2020-04-27 00:40:05.761783


In [5]:
test_query = '''
SELECT 
    'before' AS time, AVG(julianday(first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) AS avg_diff
FROM test AS t
LEFT JOIN deadlines AS d ON t.labname = d.labs
WHERE first_commit_ts < first_view_ts
UNION ALL
SELECT 
    'after' AS time, AVG(julianday(first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) AS avg_diff
FROM test AS t
LEFT JOIN deadlines AS d ON t.labname = d.labs
WHERE first_commit_ts >= first_view_ts;
'''

test_results = pd.read_sql(test_query, connection)
print(test_results)

     time  avg_diff
0  before -4.584065
1   after -4.983719


In [6]:
control_query = '''
SELECT 
    'before' AS time, 
    AVG(julianday(first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) AS avg_diff
FROM control AS c
LEFT JOIN deadlines AS d ON c.labname = d.labs
WHERE first_commit_ts < first_view_ts
UNION ALL
SELECT 
    'after' AS time, 
    AVG(julianday(first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) AS avg_diff
FROM control AS c
LEFT JOIN deadlines AS d ON c.labname = d.labs
WHERE first_commit_ts >= first_view_ts;
'''

control_results = pd.read_sql(control_query, connection)
print(control_results)


     time  avg_diff
0  before -5.045860
1   after -5.093449


In [7]:
connection.close()

In [8]:
test_before_avg = test_results[test_results['time'] == 'before']['avg_diff'].mean()
test_after_avg = test_results[test_results['time'] == 'after']['avg_diff'].mean()

control_before_avg = control_results[control_results['time'] == 'before']['avg_diff'].mean()
control_after_avg = control_results[control_results['time'] == 'after']['avg_diff'].mean()

print(f"Test group - before: {test_before_avg}, after: {test_after_avg}")
print(f"Control group - before: {control_before_avg}, after: {control_after_avg}")

if test_after_avg < test_before_avg and control_after_avg == control_before_avg:
    print("\nThe hypothesis is true: the newsfeed does affect students' behavior.")
else:
    print("\nThe hypothesis is not true: the newsfeed does not significantly affect students' behavior.")

Test group - before: -4.584065190173294, after: -4.983719459536951
Control group - before: -5.045859975708199, after: -5.093449041435424

The hypothesis is not true: the newsfeed does not significantly affect students' behavior.
